# Checking the Survival Probability

We check the spectral properties of the Perron-Frobenius operator $\hat{O} = \hat{O}_{\delta} \left( \phi, \tau \right) = \hat{P}_{\delta} \hat{U} \left( \phi, \tau \right) = \ket{\delta}\bra{\delta} \exp{ \{ -i \hat{L}(\phi) \tau \} }$ to wrap our head around the correct expression for the survival probabilty $S(n) = \vert| \hat{O}^n | \psi_0 \rangle \vert|^2$.

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import expm 
import matplotlib.pyplot as plt 

##########################################
# graph size and chirality
##########################################

num_sites = 21
target_site = num_sites // 2
#phi = 0
phi = np.pi/(2*num_sites)
#phi = np.pi/(num_sites)
tau = 1.30

##########################################
# Laplacian matrix for a 1D line (L = D - A)
##########################################

# on-site energies set to 0
diag = 0 * np.eye(num_sites)
# first-neighbor hopping
upper_diag = np.exp(1j * phi) * np.eye(num_sites, k=1)
lower_diag = np.exp(-1j * phi) * np.eye(num_sites, k=-1)

# sum
L = diag + upper_diag + lower_diag
# add periodic boundary conditions (it becomes a ring...)
L[0,-1] = np.exp(-1j * phi)
L[-1,0] = np.exp(1j * phi)

# check the form of the matrix
print(rf"Laplacian matrix L for N = {num_sites}, phi = {phi}")
df_L = pd.DataFrame(L)
pd.set_option("display.precision", 3)
display(df_L)

##########################################
# unitary evolution
##########################################
U = expm(-1j * L * tau)

print(rf"Unitary step with tau = {tau}")
df_U = pd.DataFrame(U)
pd.set_option("display.precision", 3)
display(df_U)

##########################################
# projector P = 1 - D
##########################################
P = np.eye(num_sites)
P[target_site,target_site] = 0

print(rf"Projector on the 'parallel' subspace, where |target> = |{target_site}>")
df_P = pd.DataFrame(P)
pd.set_option("display.precision", 3)
display(df_P)

##########################################
# Perron-Frobenius operator
##########################################
O = P @ U

print(rf"Perron-Frobenius (or survival) operator O = P U")
df_O = pd.DataFrame(O)
pd.set_option("display.precision", 3)
display(df_O)

### Conflicting Formulas

Now we wish to check two formulas for the survival probability:

$$ S_1(n) = \sum_{j=0}^{N-1} |\mu_j|^{2n} \langle \psi_0 | \mu_j \rangle \langle \bar{\mu}_j | \psi_0 \rangle ,$$

$$ S_2(n) = \sum_{j,k=0}^{N-1} (\mu_j^* \mu_k)^n \langle \mu_j | \mu_k \rangle \langle \psi_0 | \bar{\mu}_j \rangle  \langle \bar{\mu}_k | \psi_0 \rangle .$$

We first need some functions.

In [ ]:
from scipy.linalg import eig

##########################################
# 1) compute all eigenvalues
##########################################

def calculate_eigenvalues(matrix):
    return np.linalg.eigvals(matrix)

##########################################
# 2) look for degeneracy or modulus one
##########################################

def check_degeneracy(eigenvalues, tolerance=1e-4):
    n = len(eigenvalues)
    for i in range(n):
        for j in range(i+1,n):
            if np.isclose(eigenvalues[i],eigenvalues[j], atol=tolerance):
                return True
    return False

def check_modulus_one(eigenvalues, tolerance=1e-4):
    n = len(eigenvalues)
    for i in range(n):
        if np.isclose(np.abs(eigenvalues[i]), 1.0, atol=tolerance):
            return True
    return False

##########################################
# 3) plot complex eigenvalues
##########################################
def plot_eigenvalues(eigenvalues):
    # "zorder" defines the order in which things are drawn 
    plt.figure(figsize=(6,6))
    plt.scatter(eigenvalues.real, eigenvalues.imag, color='g', marker='o', zorder=3, label='eigenvalues')
    # axes 
    plt.axhline(0, color='black',linewidth=1, zorder=1)
    plt.axvline(0, color='black',linewidth=1, zorder=1)
    # unit circle (basically drawn with the help of polar coordinates)
    theta = np.linspace(0, 2*np.pi, 200)
    plt.plot(np.cos(theta), np.sin(theta), 'r--', alpha=0.5, zorder=2)
    # labels and proper plotting
    plt.xlabel('Re(z)')
    plt.ylabel('Im(z)')
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.axis('equal') # makes sure the ratio is 1:1
    plt.legend()
    plt.show()

##########################################
# 4) highest-modulus eigenvalue as a funct. of phi and tau
##########################################

def plot_O_leading_eigenvalue(num_sites, target_site, resolution = 600, tau_min = 0.02, tau_max = 3.00):

    # phi boundaries (tau is in the argument and defaulted to some values)
    phi_min = -np.pi/num_sites
    phi_max = +np.pi/num_sites

    # prepare blank grid
    tau_vals = np.linspace(tau_min, tau_max, resolution)
    phi_vals = np.linspace(phi_min, phi_max, resolution)
    Tau_grid, Phi_grid = np.meshgrid(tau_vals, phi_vals)
    max_mod_matrix = np.zeros_like(Tau_grid)

    # static projector
    Proj = np.eye(num_sites)
    Proj[target_site, target_site] = 0
    
    for i in range(Tau_grid.shape[0]):
        for j in range(Tau_grid.shape[1]):

            # pick values from the grid to make L (and H, and U...)
            current_tau = Tau_grid[i, j]
            current_phi = Phi_grid[i, j]
            # first-neigh.
            upper_diag_temp = np.exp(1j * current_phi) * np.eye(num_sites, k=1)
            lower_diag_temp = np.exp(-1j * current_phi) * np.eye(num_sites, k=-1)
            # periodic b.c.
            L_temp = upper_diag_temp + lower_diag_temp
            L_temp[0, -1] = np.exp(-1j * current_phi)
            L_temp[-1, 0] = np.exp(1j * current_phi)

            # O as a function of phi and tau!
            U_temp = expm(-1j * L_temp * current_tau)
            O_temp = Proj @ U_temp

            # find the greatest eigenvalue
            evals = np.linalg.eigvals(O_temp)
            max_mod_matrix[i, j] = np.max(np.abs(evals))

    plt.figure(figsize=(8,6))
    contour = plt.contour(Tau_grid, Phi_grid, max_mod_matrix, levels=100, cmap='viridis')
    #heatmap = plt.pcolormesh(Tau_grid, Phi_grid, max_mod_matrix, cmap='viridis', shading='auto')
    plt.colorbar(contour, label=f'max |\u03BC| of O')
    #plt.colorbar(heatmap, label=rf'max |\mu| of O')
    plt.xlabel('\u03C4')
    plt.ylabel('\u03C6')
    plt.show()
    
##########################################
# 5) biorthogonal decomposition
##########################################

def biorthogonal_decomposition(matrix):
    evals, left_evecs, right_evecs = eig(matrix, left=True, right=True)
    # normalization
    # i is the index of the eigenvector
    # : runs over its components
    for i in range(len(evals)): 
        overlap = np.vdot(left_evecs[:,i], right_evecs[:,i])
        if np.abs(overlap) > 1e-15: # don't wanna be dividing by 0...
            left_evecs[:, i] = left_evecs[:, i] / np.conj(overlap)

    return evals, left_evecs, right_evecs

##########################################
# 6) S_1(n), S_2(n), and the old good || O^n | psi_0 > || ^ 2
##########################################

import warnings

def plot_survival_probabilities(O_matrix, num_steps = 100, psi_0 = None, tolerance=1e-8, ymin=0.001, ymax=0.6, logscale=False):

    N = O_matrix.shape[0]

    # if not otherwise specified, the initial state is localized at |0> 
    if psi_0 is None: 
        psi_0 = np.zeros(N, dtype=complex)
        psi_0[0] = 1.0

    mu, L_evecs, R_evecs = biorthogonal_decomposition(O_matrix)

    n_vals = np.arange(1, num_steps + 1) # given as an input to the function

    # formulas employing decompositions
    S1_vals = np.zeros(num_steps, dtype=float)
    S2_vals = np.zeros(num_steps, dtype=float)

    # brute force calculation from the matrix form (actually more efficient)
    S_from_matrix_vals = np.zeros(num_steps, dtype=float)
    psi_n = psi_0.copy()


    for idx, n in enumerate(n_vals):

        ############ S_1(n) ############  
         
        s1 = 0.0 + 0.0j

        for j in range(N):
            factor1 = np.abs(mu[j])**(2*n)
            factor2 = np.vdot(psi_0, R_evecs[:, j]) # <psi_0|mu_j>
            factor3 = np.vdot(L_evecs[:, j], psi_0) # <mu_bar_j|psi_0>
            s1 += factor1 * factor2 * factor3

        if np.abs(np.imag(s1)) > tolerance:
            warnings.warn(f"S_1(n={n}) has a non-negligible Im part = {np.imag(s1):.2e}")

        S1_vals[idx] = np.real(s1)

        ############ S_2(n) ############  

        s2 = 0.0 + 0.0j

        for j in range(N):
            for k in range(N):
                factor1 = (np.conj(mu[j])*mu[k])**n
                factor2 =  np.vdot(R_evecs[:, j], R_evecs[:, k]) # <mu_j|mu_k>                 
                factor3 =  np.vdot(psi_0, L_evecs[:, j]) # <psi_0|mu_bar_j> 
                factor4 =  np.vdot(L_evecs[:, k], psi_0) # <mu_bar_k|psi_0>
                s2 += factor1 * factor2 * factor3 * factor4

        if np.abs(np.imag(s2)) > tolerance:
            warnings.warn(f"S_2(n={n}) has a non-negligible Im part = {np.imag(s2):.2e}")
        
        S2_vals[idx] = np.real(s2)

        ############ || O^n | psi_0 > || ^ 2 ############  

        psi_n = O_matrix @ psi_n
        S_from_matrix_vals[idx] = np.real(np.vdot(psi_n, psi_n)) # sum(np.abs(psi_n)**2)


    # plot for colorblind
    plt.figure(figsize=(15,9))
    plt.plot(n_vals, S1_vals, label='$S_1(n)$', color='#0072B2', linestyle='-', marker='o', markersize = 4, alpha=0.8)
    plt.plot(n_vals, S2_vals, label='$S_2(n)$', color='#D55E00', linestyle='--', marker='x', alpha=0.8)
    plt.plot(n_vals, S_from_matrix_vals, label=r'$|| O^n | \psi_0 \rangle ||^2$', color='#009E73', linestyle=':', marker='s', markersize = 3, alpha=0.8)
    plt.xlabel('step $n$')
    plt.ylabel('survival probability')
    plt.ylim(ymin,ymax)
    if logscale:
        plt.yscale('log')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.show()
            



### Numerical Comparison

We now wanna compare $S_1$ and $S_2$ as a function of $n$, to see how different they are and if they converge to the same value as $n \to \infty$.

In [ ]:
# 1) get the eigenvalues of the P.-F. operator

eigenvalues_O = calculate_eigenvalues(O)

print(f"Parameters:")
print(rf"N = {num_sites}, phi = {100.0*phi/(np.pi/num_sites)}% of pi/N, tau = {tau}")

print(f"List of eigenvalues:")
print(eigenvalues_O)

# 2) check degeneracy and modulus 1

has_deg = check_degeneracy(eigenvalues_O)
print(f"Is the spectrum of O degenerate? {has_deg}")

has_mod_one = check_modulus_one(eigenvalues_O)
print(f"Any modulus one eigenvalue? {has_mod_one}")

# 3) plot on the complex plane

plot_eigenvalues(eigenvalues_O)

In [ ]:
# 4) plot the eigenvalues of O as a function of phi and tau!
plot_O_leading_eigenvalue(num_sites, num_sites//2)

In [ ]:
# 5) carry out the biorthogonal decomposition
evals_B_decomp, L_Evecs, R_Evecs = biorthogonal_decomposition(O)
# check for biorthogonality
biorthogonality_check = L_Evecs.conj().T @ R_Evecs
print("L^dagger @ R (should be ~ Identity):\n")
#df_BC = pd.DataFrame(biorthogonality_check)
df_BC = pd.DataFrame(np.round(biorthogonality_check, 6))
pd.set_option("display.precision", 3)
display(df_BC)

In [ ]:
# 6) compare S_1 and S_2
plot_survival_probabilities(O, num_steps = 125, ymin=0.05, ymax=0.35)